#DEEP Practical Week 7: Diffusion models and flow matching

Last week you trained GANs, which turn noise into data in a single step. This week you'll build models that turn noise into data in many small steps. Start small, with 2D points you can see, and work up to generating shapes and then photographs.

1. **A diffusion model in 2D** (video 1). Train a DDPM on last week's spiral and watch samples travel from noise to data. Runs on the CPU in under a minute.
2. **Flow matching in 2D** (video 3). Change three lines and get a model that needs far fewer steps.
3. **A diffusion model for shapes** (videos 1 to 3). Generate shape outlines, with classifier-free guidance to choose the shape (video 2). Use a GPU runtime for this one: training takes a couple of minutes.
4. **Play with Stable Diffusion** (video 2). Text-to-image with a real pretrained latent diffusion model, and a look inside its denoising loop.
5. **Optional, highly recommended for Masters students: LoRA** (video 5). Teach the shape model a new shape from just 20 examples.

Everyone should do exercises 1 to 3. Exercise 4 is fun and needs no coding.

##1. A diffusion model in 2D

Our "dataset" is the noisy spiral from last week's GAN exercise. Each data point is just a 2D point. As with the GAN, the model will learn to turn 2D Gaussian noise into points on the spiral. This time it won't be done in one step but by gradually removing noise.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import math
import matplotlib.pyplot as plt

torch.manual_seed(0)

def sample_spiral(n, noise=0.05):
    # Points along a spiral (more points further out, so the density is roughly even), plus Gaussian noise
    t = torch.sqrt(torch.rand(n)) * 3 * np.pi + 0.5
    x = torch.stack([t * torch.cos(t), t * torch.sin(t)], 1) / (3 * np.pi)
    return x + noise * torch.randn(n, 2)

real = sample_spiral(2000)
plt.figure(figsize=(5, 5))
plt.scatter(real[:, 0], real[:, 1], s=2, c="gray")
plt.gca().set_aspect("equal")
plt.title("Real data: noisy spiral")
plt.show()

###The forward process

The forward process gradually destroys the data by mixing it with Gaussian noise. From video 1, we can jump straight from a clean point $\mathbf{x}_0$ to its noisy version at step $t$:

$$\mathbf{x}_t=\sqrt{\bar{\alpha}_t}\,\mathbf{x}_0+\sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\epsilon},\qquad \boldsymbol{\epsilon}\sim\mathrm{N}(\mathbf{0},\mathbf{I})$$

where $\alpha_t = 1-\beta_t$ and $\bar{\alpha}_t=\prod_{s=1}^t\alpha_s$. Images need around 1,000 steps; for 2D points $T=100$ is plenty. The code below sets up the variance schedule $\beta_t$, increasing linearly. (In code we count steps from 0 to $T-1$.)

**To do**: complete `add_noise`. Then run the next cell to see the spiral dissolve into noise.

In [ ]:
T = 100
betas = torch.linspace(1e-4, 0.15, T)          # variance schedule beta_t
alphas = 1 - betas                             # alpha_t
alpha_bars = torch.cumprod(alphas, dim=0)      # alpha-bar_t = product of alphas up to t
print('alpha-bar at the last step: {:.5f}'.format(alpha_bars[-1].item()))

In [ ]:
def add_noise(x0, t, eps):
    # x0: batch of clean points (B x 2), t: batch of time steps (B,) integers in 0..T-1, eps: noise (B x 2)
    # Return the noisy points x_t
    # To do
    return None

In [ ]:
x0 = sample_spiral(2000)
eps = torch.randn_like(x0)
figure, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, t in zip(axes, [0, 10, 25, 50, 99]):
    xt = add_noise(x0, torch.full((2000,), t), eps)
    ax.scatter(xt[:, 0], xt[:, 1], s=2)
    ax.set_title('t = {}'.format(t)); ax.set_aspect('equal'); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.show()

###The denoising network

For images, the network that predicts the noise is a U-Net. For 2D points, a small MLP will do. The network needs to know how noisy its input is, so it also takes the time step $t$. Rather than feeding in $t$ as a single number, we turn it into a vector of sines and cosines at different frequencies (a *sinusoidal embedding*, the same trick transformers use for positions, which you'll meet next week). This makes it easy for the network to behave differently at different noise levels.

In [ ]:
def time_embedding(t, dim=16, max_period=100.0):
    # t: (B,) tensor of times. Returns B x dim features: sines and cosines at dim/2 frequencies
    freqs = torch.exp(-math.log(max_period) * torch.arange(dim // 2) / (dim // 2))
    args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)

class Denoiser2D(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 + 16, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2))
    def forward(self, x, t):
        # x: B x 2 noisy points, t: (B,) times. Output: B x 2 (the predicted noise)
        return self.net(torch.cat([x, time_embedding(t)], dim=1))

###Training

Each training step follows the recipe from video 1:

1. Take a batch of real points $\mathbf{x}_0$.
2. Choose a random time step $t$ for each point, and random noise $\boldsymbol{\epsilon}$.
3. Make the noisy points $\mathbf{x}_t$ using `add_noise`.
4. The loss is the mean squared error between the true noise $\boldsymbol{\epsilon}$ and the network's prediction $\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$.

**To do**: complete the loss in the training loop below. It should take under a minute on the CPU (you don't need a GPU for exercises 1 and 2).

In [ ]:
model = Denoiser2D()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
num_steps = 20000
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, num_steps)
batch_size = 256
for step in range(num_steps):
    x0 = sample_spiral(batch_size)
    # To do: sample t and eps, make xt and compute the loss
    loss = None
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    scheduler.step()
    if (step + 1) % 2000 == 0:
        print('step {}: loss {:.4f}'.format(step + 1, loss.item()))

Notice that the loss doesn't go anywhere near zero. Why not? (Think about what the network would have to know to predict the noise exactly at $t=99$.)

###Sampling

To generate, we start from pure noise $\mathbf{x}_T \sim \mathrm{N}(\mathbf{0},\mathbf{I})$ and go backwards one step at a time. At each step the network predicts the noise, we remove the right amount of it, and then add a little fresh noise (except at the last step):

$$\mathbf{x}_{t-1}=\frac{1}{\sqrt{\alpha_t}}\left(\mathbf{x}_t-\frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\,\boldsymbol{\epsilon}_\theta(\mathbf{x}_t,t)\right)+\sqrt{\beta_t}\,\mathbf{z},\qquad \mathbf{z}\sim\mathrm{N}(\mathbf{0},\mathbf{I})$$

Read through the code and check you can match it to the equation. It keeps the whole path of each sample so we can draw the trajectories.

In [ ]:
@torch.no_grad()
def ddpm_sample(model, n, fresh_noise=True):
    x = torch.randn(n, 2)
    path = [x]
    for t in reversed(range(T)):
        eps_pred = model(x, torch.full((n,), t))
        x = (x - betas[t] / (1 - alpha_bars[t]).sqrt() * eps_pred) / alphas[t].sqrt()
        if t > 0 and fresh_noise:
            x = x + betas[t].sqrt() * torch.randn_like(x)
        path.append(x)
    return torch.stack(path)        # (T+1) x n x 2

def show_samples(path, title, n_paths=5):
    figure, axes = plt.subplots(1, 2, figsize=(12, 6))
    real = sample_spiral(2000)
    axes[0].scatter(real[:, 0], real[:, 1], s=2, c='gray', label='real')
    axes[0].scatter(path[-1, :, 0], path[-1, :, 1], s=2, c='tab:orange', label='generated')
    axes[0].legend(); axes[0].set_title(title)
    for i in range(n_paths):
        axes[1].plot(path[:, i, 0], path[:, i, 1], lw=0.8)
        axes[1].scatter(path[0, i, 0], path[0, i, 1], c='k', s=15)
        axes[1].scatter(path[-1, i, 0], path[-1, i, 1], c='r', s=15)
    axes[1].set_title('Paths from noise (black) to samples (red)')
    for ax in axes: ax.set_aspect('equal'); ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    plt.show()

path_ddpm = ddpm_sample(model, 2000)
show_samples(path_ddpm, 'DDPM, {} steps'.format(T))

**Things to look at and try**:

1. Does the model cover the whole spiral, including the inner end? Compare with your GAN from last week.
2. Are there generated points *between* the arms of the spiral? Why might that happen? (Hint: what does a single small MLP find hard to represent?)
3. Look at the paths. Why are they so jagged?
4. Run `ddpm_sample(model, 2000, fresh_noise=False)`, i.e. drop the fresh noise. What happens to the samples, and why?

In [ ]:
path_nonoise = ddpm_sample(model, 2000, fresh_noise=False)
show_samples(path_nonoise, 'DDPM without fresh noise')

##2. Flow matching in 2D

Video 3 introduced flow matching. Instead of the DDPM noise schedule we join each data point to a noise sample by a straight line, with time $t$ now running continuously from 0 (data) to 1 (noise):

$$\mathbf{x}_t=(1-t)\,\mathbf{x}_0+t\,\boldsymbol{\epsilon}$$

and the network learns the velocity along that line, $\boldsymbol{\epsilon}-\mathbf{x}_0$:

$$L=\big\|\mathbf{v}_\theta(\mathbf{x}_t,t)-(\boldsymbol{\epsilon}-\mathbf{x}_0)\big\|^2$$

We can reuse exactly the same network. Since our time embedding was designed for times between 0 and 100, we pass in `100 * t`.

**To do**:

1. Complete the training loop. Only three lines differ from exercise 1: how $t$ is sampled, how $\mathbf{x}_t$ is made, and the target.
2. Complete the Euler sampler. Generation follows the velocity backwards from $t=1$ to $t=0$: $\mathbf{x}_{t-\Delta t}=\mathbf{x}_t-\Delta t\,\mathbf{v}_\theta(\mathbf{x}_t,t)$, with no fresh noise.

In [ ]:
model_fm = Denoiser2D()
optimiser = torch.optim.Adam(model_fm.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, num_steps)
for step in range(num_steps):
    x0 = sample_spiral(batch_size)
    # To do: sample t (continuous, between 0 and 1) and eps, make xt and compute the loss
    loss = None
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    scheduler.step()
    if (step + 1) % 2000 == 0:
        print('step {}: loss {:.4f}'.format(step + 1, loss.item()))

@torch.no_grad()
def euler_sample(model, n, steps=20):
    x = torch.randn(n, 2)
    path = [x]
    for k in range(steps, 0, -1):
        t = torch.full((n,), k / steps)       # current time, from 1 down to 1/steps
        # To do: one Euler step backwards in time
        path.append(x)
    return torch.stack(path)

In [ ]:
path_fm = euler_sample(model_fm, 2000, steps=20)
show_samples(path_fm, 'Flow matching, 20 Euler steps', n_paths=15)

**Things to look at and try**:

1. Compare the paths with the DDPM ones. Why are they so much smoother?
2. Try `steps` = 1, 2, 5 and 50. How few steps can you get away with? With 1 step, where do all the samples end up, and why?
3. The loss for flow matching is much larger than for DDPM. Does that mean it's a worse model?

In [ ]:
figure, axes = plt.subplots(1, 4, figsize=(18, 4.5))
real = sample_spiral(2000)
for ax, steps in zip(axes, [1, 2, 5, 50]):
    p = euler_sample(model_fm, 2000, steps)
    ax.scatter(real[:, 0], real[:, 1], s=2, c='gray')
    ax.scatter(p[-1, :, 0], p[-1, :, 1], s=2, c='tab:orange')
    ax.set_title('{} steps'.format(steps)); ax.set_aspect('equal'); ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
plt.show()

##3. A diffusion model for shapes

Now something more interesting than points: shapes. Each shape is the outline of a circle, triangle, square, star, heart or cross, stored as 64 points around its boundary. So one data item is a $2\times 64$ tensor: two "channels" (the $x$ and $y$ coordinates) along a sequence of 64 positions. That's exactly like a 1D image, so we can denoise it with a 1D CNN.

The shapes are randomly rotated, scaled, stretched and moved, and the outline can start at any of its points, so there's plenty of variety. **Switch to a GPU runtime for this exercise.**

In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

CLASSES = ['circle', 'triangle', 'square', 'star', 'heart', 'cross']
N = 64   # points per outline

def resample(path, n):
    # path: M x 2 closed polyline (last point == first). Return n points equally spaced along it
    seg = path[1:] - path[:-1]
    L = seg.norm(dim=1); cum = torch.cat([torch.zeros(1), L.cumsum(0)])
    s = torch.linspace(0, float(cum[-1]), n + 1)[:-1]
    idx = torch.searchsorted(cum, s, right=True).clamp(1, len(L)) - 1
    frac = ((s - cum[idx]) / L[idx]).unsqueeze(1)
    return path[idx] + frac * seg[idx]

def template(name):
    t = torch.linspace(0, 2 * math.pi, 400)
    if name == 'circle':
        p = torch.stack([t.cos(), t.sin()], 1)
    elif name == 'heart':
        p = torch.stack([16 * t.sin() ** 3, 13 * t.cos() - 5 * (2 * t).cos() - 2 * (3 * t).cos() - (4 * t).cos()], 1) / 16
    else:
        if name == 'cross':
            w = 0.35
            p = torch.tensor([[w, 1], [w, w], [1, w], [1, -w], [w, -w], [w, -1], [-w, -1], [-w, -w], [-1, -w], [-1, w], [-w, w], [-w, 1]])
        else:
            n, r = {'triangle': (3, [1.]), 'square': (4, [1.]), 'star': (10, [1., 0.45])}[name]
            a = torch.arange(n) * 2 * math.pi / n + (math.pi / 4 if name == 'square' else math.pi / 2)
            r = torch.tensor(r * (n // len(r)))
            p = torch.stack([r * a.cos(), r * a.sin()], 1)
    p = resample(torch.cat([p, p[:1]]), N)
    return p - p.mean(0)

TEMPLATES = torch.stack([template(c) for c in CLASSES])     # 6 x N x 2

def sample_shapes(batch, classes=None):
    # Returns x: batch x 2 x N outlines and y: (batch,) class labels.
    # classes: optional (batch,) tensor of the classes you want
    y = torch.randint(len(CLASSES), (batch,)) if classes is None else classes
    x = TEMPLATES[y]
    shift = torch.randint(N, (batch,))                                   # random starting point
    idx = (torch.arange(N)[None] + shift[:, None]) % N
    x = torch.gather(x, 1, idx[..., None].expand(-1, -1, 2))
    ang = (torch.rand(batch) * 2 - 1) * math.pi / 6                      # rotate by up to 30 degrees
    R = torch.stack([torch.stack([ang.cos(), -ang.sin()], 1), torch.stack([ang.sin(), ang.cos()], 1)], 1)
    scale = 0.6 + 0.4 * torch.rand(batch, 1, 1)
    stretch = torch.stack([1 + 0.25 * (torch.rand(batch) * 2 - 1), torch.ones(batch)], 1)[:, None]
    x = (x * stretch) @ R.transpose(1, 2) * scale + 0.2 * (torch.rand(batch, 1, 2) * 2 - 1)
    return x.transpose(1, 2).contiguous(), y

def draw(ax, outline, colour='tab:blue', points=False, lim=1.5):
    p = outline.detach().cpu().T                  # N x 2
    closed = torch.cat([p, p[:1]])
    ax.plot(closed[:, 0], closed[:, 1], '-', color=colour, lw=1.2)
    if points: ax.scatter(p[:, 0], p[:, 1], s=4, color=colour)
    ax.set_aspect('equal'); ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.axis('off')

x, y = sample_shapes(24)
figure, axes = plt.subplots(3, 8, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    draw(ax, x[i], points=True); ax.set_title(CLASSES[y[i]])
plt.show()
print(x.shape)

###The network: a 1D CNN with circular padding

The denoiser below is a 1D CNN. A few things to notice:

- **Circular padding**: an outline is closed, so the point after the last one is the first one. `padding_mode='circular'` makes the convolution wrap around the ends instead of padding with zeros.
- **Dilated convolutions** (dilation 1, 2, 4, 8, 16): each layer looks at points further apart, so after a few layers every output can see the whole outline. This does the job that the down-sampling does in a U-Net.
- **Time and class embeddings** are added inside every residual block, so every layer knows how noisy the input is and which shape we want.
- The class embedding has **one extra entry**, number 6, meaning "no class". We need it for classifier-free guidance (video 2).
- The final layer is initialised to zero, so the untrained network predicts zero. This makes the start of training more stable.

In [ ]:
def timestep_embedding(t, dim=64):
    half = dim // 2
    freqs = torch.exp(-math.log(1000.0) * torch.arange(half, device=t.device) / half)
    a = t[:, None].float() * freqs[None]
    return torch.cat([a.sin(), a.cos()], 1)

class ResBlock(nn.Module):
    def __init__(self, ch, emb_dim, dilation):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, ch)
        self.conv1 = nn.Conv1d(ch, ch, 3, padding=dilation, dilation=dilation, padding_mode='circular')
        self.emb = nn.Linear(emb_dim, ch)
        self.norm2 = nn.GroupNorm(8, ch)
        self.conv2 = nn.Conv1d(ch, ch, 3, padding=dilation, dilation=dilation, padding_mode='circular')
        self.act = nn.SiLU()
    def forward(self, x, e):
        h = self.conv1(self.act(self.norm1(x)))
        h = h + self.emb(e)[:, :, None]           # add the time + class embedding at every position
        h = self.conv2(self.act(self.norm2(h)))
        return x + h

class ShapeDenoiser(nn.Module):
    def __init__(self, n_classes=6, ch=64, emb_dim=128, dilations=(1, 2, 4, 8, 16, 1, 2, 4, 8, 16)):
        super().__init__()
        self.time_mlp = nn.Sequential(nn.Linear(64, emb_dim), nn.SiLU(), nn.Linear(emb_dim, emb_dim))
        self.class_emb = nn.Embedding(n_classes + 1, emb_dim)   # last entry = "no class"
        self.null_class = n_classes
        self.inp = nn.Conv1d(2, ch, 3, padding=1, padding_mode='circular')
        self.blocks = nn.ModuleList([ResBlock(ch, emb_dim, d) for d in dilations])
        self.out = nn.Sequential(nn.GroupNorm(8, ch), nn.SiLU(), nn.Conv1d(ch, 2, 3, padding=1, padding_mode='circular'))
        nn.init.zeros_(self.out[-1].weight); nn.init.zeros_(self.out[-1].bias)
    def forward(self, x, t, y):
        # x: B x 2 x N noisy outlines, t: (B,) times, y: (B,) class labels (6 = no class)
        e = self.time_mlp(timestep_embedding(t)) + self.class_emb(y)
        h = self.inp(x)
        for block in self.blocks:
            h = block(h, e)
        return self.out(h)

print(sum(p.numel() for p in ShapeDenoiser().parameters()), 'parameters')

###Training with classifier-free guidance in mind

Training is the DDPM loss from exercise 1 with $T=200$ steps, plus the class label. To make classifier-free guidance possible (video 2), the model must also learn to denoise *without* knowing the class. So for a random 10% of each batch we replace the label with the "no class" label, 6.

The function also supports flow matching (`mode='fm'`), which you'll use below, and training on only some classes (`classes`), which you'll need in exercise 5.

**To do**: complete the label dropout line, then train. It takes 2 to 3 minutes on a GPU.

In [ ]:
T_shape = 200
betas_s = torch.linspace(1e-4, 0.08, T_shape).to(device)
alphas_s = 1 - betas_s
alpha_bars_s = torch.cumprod(alphas_s, 0)

def train_shapes(model, steps=3000, mode='ddpm', classes=None, batch=256, lr=2e-3):
    # classes: optional list of class numbers to train on (default: all)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.05)
    allowed = torch.arange(len(CLASSES)) if classes is None else torch.tensor(classes)
    for it in range(steps):
        x0, y = sample_shapes(batch, classes=allowed[torch.randint(len(allowed), (batch,))])
        x0, y = x0.to(device), y.to(device)
        # To do: replace a random 10% of the labels with model.null_class (hint: torch.rand and torch.where)
        y = y
        eps = torch.randn_like(x0)
        if mode == 'ddpm':
            t = torch.randint(0, T_shape, (batch,), device=device)
            a = alpha_bars_s[t][:, None, None]
            xt = a.sqrt() * x0 + (1 - a).sqrt() * eps
            loss = ((model(xt, t, y) - eps) ** 2).mean()
        else:   # flow matching: continuous t in [0, 1]; the time embedding expects 0..T_shape
            t = torch.rand(batch, device=device)
            xt = (1 - t[:, None, None]) * x0 + t[:, None, None] * eps
            loss = ((model(xt, t * T_shape, y) - (eps - x0)) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if (it + 1) % 500 == 0:
            print('step {}: loss {:.4f}'.format(it + 1, loss.item()))
    return model

torch.manual_seed(0)
shape_model = train_shapes(ShapeDenoiser().to(device))

###Sampling with classifier-free guidance

From video 2, classifier-free guidance runs the network twice at every step, once with the class and once without, and pushes the prediction away from the unconditional one:

$$\tilde{\boldsymbol{\epsilon}}=(1+w)\,\boldsymbol{\epsilon}_\theta(\mathbf{x}_t,t,y)-w\,\boldsymbol{\epsilon}_\theta(\mathbf{x}_t,t,\text{None})$$

where $w$ is the guidance scale ($w=0$ means no guidance).

**To do**: complete `guided_prediction`. The sampler below then uses it in exactly the DDPM update from exercise 1, and keeps a few snapshots so we can watch a shape form.

In [ ]:
def guided_prediction(model, x, t, y, w):
    # To do: return (1 + w) * prediction with class y  -  w * prediction with the "no class" label
    return None

In [ ]:
@torch.no_grad()
def sample_shape_ddpm(model, y, w=2.0, keep=()):
    x = torch.randn(len(y), 2, N, device=device)
    snapshots = {}
    for t in reversed(range(T_shape)):
        if t in keep: snapshots[t] = x.clone()
        eps_pred = guided_prediction(model, x, torch.full((len(y),), t, device=device), y, w)
        x = (x - betas_s[t] / (1 - alpha_bars_s[t]).sqrt() * eps_pred) / alphas_s[t].sqrt()
        if t > 0: x = x + betas_s[t].sqrt() * torch.randn_like(x)
    return x, snapshots

y = torch.arange(len(CLASSES), device=device).repeat_interleave(6)   # 6 of each class
samples, _ = sample_shape_ddpm(shape_model.eval(), y)
figure, axes = plt.subplots(6, 6, figsize=(10, 10))
for i, ax in enumerate(axes.T.flat):
    draw(ax, samples[i])
for j in range(6): axes[0, j].set_title(CLASSES[j])
plt.show()

# Watch three shapes form
keep = [199, 150, 100, 60, 30, 10, 0]
final, snaps = sample_shape_ddpm(shape_model, torch.tensor([3, 4, 5], device=device), keep=keep)
figure, axes = plt.subplots(3, len(keep) + 1, figsize=(16, 6))
for i in range(3):
    for j, t in enumerate(keep):
        draw(axes[i, j], snaps[t][i], points=True, lim=2.5); axes[0, j].set_title('t = {}'.format(t))
    draw(axes[i, -1], final[i]); axes[0, -1].set_title('result')
plt.show()

**Things to try**:

1. Generate a row of hearts with guidance scales $w = 0, 1, 3$ and $8$. What changes? What goes wrong if $w$ is too large?
2. Use the "no class" label (6) for every sample, so you're sampling the unconditional model. Do you get a mixture of all the shapes?
3. **Flow matching for shapes.** Train a second model with `train_shapes(ShapeDenoiser().to(device), mode='fm')`. Write an Euler sampler (like exercise 2, but with `guided_prediction` and remembering to pass `t * T_shape` as the time). How many steps do you need? What do you get with a single step?

In [ ]:
# Solution: guidance scale
figure, axes = plt.subplots(4, 8, figsize=(14, 7))
for r, w in enumerate([0, 1, 3, 8]):
    s, _ = sample_shape_ddpm(shape_model, torch.full((8,), 4, device=device), w=w)
    for j in range(8): draw(axes[r, j], s[j])
    axes[r, 0].set_title('w = {}'.format(w), loc='left')
plt.show()

# Solution: unconditional samples
s, _ = sample_shape_ddpm(shape_model, torch.full((16,), shape_model.null_class, device=device), w=0)
figure, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat): draw(ax, s[i])
plt.show()

In [ ]:
# Solution: flow matching for shapes
torch.manual_seed(0)
shape_model_fm = train_shapes(ShapeDenoiser().to(device), mode='fm')

@torch.no_grad()
def sample_shape_fm(model, y, w=2.0, steps=20):
    x = torch.randn(len(y), 2, N, device=device)
    for k in range(steps, 0, -1):
        t = torch.full((len(y),), k / steps, device=device)
        x = x - (1 / steps) * guided_prediction(model, x, t * T_shape, y, w)
    return x

y = torch.arange(len(CLASSES), device=device)
figure, axes = plt.subplots(5, 6, figsize=(10, 9))
for r, steps in enumerate([1, 2, 4, 8, 20]):
    torch.manual_seed(4)
    s = sample_shape_fm(shape_model_fm.eval(), y, steps=steps)
    for j in range(6): draw(axes[r, j], s[j])
    axes[r, 0].set_title('{} steps'.format(steps), loc='left')
plt.show()

##4. Play with a pretrained diffusion model (Stable Diffusion)

First we need to install some dependencies and import. This takes a few minutes to run so take a break while it's running!

Stable Diffusion is the latent diffusion model from video 2. A VAE compresses each 512 x 512 image to a 4 x 64 x 64 latent, a U-Net does the denoising in that latent space, and a CLIP text encoder turns your prompt into the conditioning. It uses classifier-free guidance, exactly as you implemented for the shapes. Make sure you're using a GPU runtime.

In [ ]:
!pip install -U diffusers
!pip install transformers scipy ftfy accelerate

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained("sd-legacy/stable-diffusion-v1-5", torch_dtype=torch.float16)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipe = pipe.to(device)

Now it's time to play! This diffusion implementation comes with a nice end to end text-to-image pipeline. You simply need to provide a text prompt and it will generate an image. You will get a different result every time as the starting random noise will be different. Experiment with different prompts.

In [ ]:
prompt = "a photograph of an astronaut riding a horse"
image = pipe(prompt).images[0]  # image here is in PIL format

# directly display the generated image
image

`pipe` can take various other option parameters. Try playing with `num_inference_steps` which controls how many diffusion steps are used (fewer is faster but lower quality), `height` and `width` which control the size of the output and `guidance_scale` which controls how much weight it gives to your text prompt.

Let's now dig a little deeper and see a denoising loop in action. First, some setup:

In [ ]:
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, UNet2DConditionModel, PNDMScheduler
from diffusers import LMSDiscreteScheduler

# 1. Load the autoencoder model which will be used to decode the latents into image space.
vae = AutoencoderKL.from_pretrained("sd-legacy/stable-diffusion-v1-5", subfolder="vae")

# 2. Load the tokenizer and text encoder to tokenize and encode the text.
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")

# 3. The UNet model for generating the latents.
unet = UNet2DConditionModel.from_pretrained("sd-legacy/stable-diffusion-v1-5", subfolder="unet")

scheduler = LMSDiscreteScheduler.from_pretrained("sd-legacy/stable-diffusion-v1-5", subfolder="scheduler")

vae = vae.to(device)
text_encoder = text_encoder.to(device)
unet = unet.to(device)

Prepare our inputs:

In [ ]:
prompt = ["a photograph of an astronaut riding a horse"]

height = 512                        # default height of Stable Diffusion
width = 512                         # default width of Stable Diffusion

num_inference_steps = 100            # Number of denoising steps

guidance_scale = 7.5                # Scale for classifier-free guidance

generator = torch.manual_seed(32)   # Seed generator to create the inital latent noise

batch_size = 1

text_input = tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")

with torch.no_grad():
  text_embeddings = text_encoder(text_input.input_ids.to(device))[0]

max_length = text_input.input_ids.shape[-1]
uncond_input = tokenizer(
    [""] * batch_size, padding="max_length", max_length=max_length, return_tensors="pt"
)
with torch.no_grad():
  uncond_embeddings = text_encoder(uncond_input.input_ids.to(device))[0]

text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

# Random initial latents
latents = torch.randn(
  (batch_size, unet.in_channels, height // 8, width // 8),
  generator=generator,
)
latents = latents.to(device)

scheduler.set_timesteps(num_inference_steps)
latents = latents * scheduler.init_noise_sigma

Now we are ready to run the denoising loop:

In [ ]:
from tqdm.auto import tqdm
from torch import autocast

for t in tqdm(scheduler.timesteps):
  # expand the latents if we are doing classifier-free guidance to avoid doing two forward passes.
  latent_model_input = torch.cat([latents] * 2)

  latent_model_input = scheduler.scale_model_input(latent_model_input, t)

  # predict the noise residual
  with torch.no_grad():
    noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings).sample

  # perform guidance
  noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
  noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

  # compute the previous noisy sample x_t -> x_t-1
  latents = scheduler.step(noise_pred, t, latents).prev_sample

Visualise the result:

In [ ]:
from PIL import Image

# scale and decode the image latents with vae
latents = 1 / 0.18215 * latents

with torch.no_grad():
  image = vae.decode(latents).sample

image = (image / 2 + 0.5).clamp(0, 1)
image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
images = (image * 255).round().astype("uint8")
pil_images = [Image.fromarray(image) for image in images]
pil_images[0]

**To do**: At any point in the loop, you could convert the noisy latents to an image using the VAE decoder. Try this out. You should be able to visualise the result gradually transitioning from noise to the final image.

##5. Optional: teach the shape model a new shape with LoRA

Video 5 introduced LoRA: freeze a pretrained model and learn a low-rank update $\Delta W = BA$ for some of its layers. Here you'll pretrain a shape model that has **never seen a heart**, then teach it hearts from only 20 examples, training only a small number of LoRA parameters.

First, pretrain without hearts (class 4). This takes another 2 to 3 minutes. Asking this model for hearts should give nonsense.

In this exercise we sample **without guidance** ($w=0$). Guidance relies on the model's "no class" predictions, and you'll see that fine-tuning can damage those too.

In [ ]:
HEART = CLASSES.index('heart')
torch.manual_seed(0)
base_model = train_shapes(ShapeDenoiser().to(device), classes=[c for c in range(len(CLASSES)) if c != HEART])
torch.save(base_model.state_dict(), 'base_no_hearts.pt')

def show_all_classes(model, title, w=0.0):
    torch.manual_seed(3)
    s, _ = sample_shape_ddpm(model.eval(), torch.arange(len(CLASSES), device=device).repeat_interleave(4), w=w)
    figure, axes = plt.subplots(1, 24, figsize=(24, 1.5))
    for i, ax in enumerate(axes): draw(ax, s[i], colour='tab:orange' if i // 4 == HEART else 'tab:blue')
    figure.suptitle(title, x=0.1, ha='left')
    plt.show()

show_all_classes(base_model, 'Pretrained without hearts')

###A LoRA layer for convolutions

A `Conv1d` weight has shape `out_channels x in_channels x kernel_size`. Treat it as a matrix with `out_channels` rows and `in_channels * kernel_size` columns, so $B$ is `out_channels x r` and $A$ is `r x (in_channels * kernel_size)`.

**To do**: complete `LoRAConv1d`. It should:

- keep the original convolution and freeze its parameters (`requires_grad = False`),
- create `A` with small random values (e.g. `torch.randn(...) / math.sqrt(in_channels * kernel_size)`) and `B` full of zeros, both as `nn.Parameter`s, so that $BA=0$ at the start,
- in `forward`, compute `delta = (self.B @ self.A).view_as(self.conv.weight) * self.scale` and apply the convolution with weight `self.conv.weight + delta`. Use `self.conv._conv_forward(x, weight, self.conv.bias)`, which applies a convolution with a given weight but keeps the layer's padding and dilation settings.

In [ ]:
class LoRAConv1d(nn.Module):
    def __init__(self, conv, r=4, alpha=4):
        super().__init__()
        self.conv = conv
        self.scale = alpha / r
        # To do

    def forward(self, x):
        # To do
        return None

The next cell loads the pretrained model, freezes everything, and wraps both convolutions in every residual block with your LoRA layer. The class embedding also stays trainable, because the model has never learnt what label 4 means. It then fine-tunes on a fixed set of **20 hearts**. Look at the number of trainable parameters.

In [ ]:
def make_lora_model(r=4):
    model = ShapeDenoiser().to(device)
    model.load_state_dict(torch.load('base_no_hearts.pt'))
    for p in model.parameters():
        p.requires_grad = False
    for block in model.blocks:
        block.conv1 = LoRAConv1d(block.conv1, r)
        block.conv2 = LoRAConv1d(block.conv2, r)
    model.class_emb.weight.requires_grad = True
    return model.to(device)

torch.manual_seed(5)
few_hearts, _ = sample_shapes(20, classes=torch.full((20,), HEART))
few_hearts = few_hearts.to(device)

def finetune(model, steps=800, replay=False, lr=1e-3):
    params = [p for p in model.parameters() if p.requires_grad]
    print('trainable parameters: {:,} of {:,}'.format(sum(p.numel() for p in params), sum(p.numel() for p in model.parameters())))
    opt = torch.optim.Adam(params, lr=lr)
    for it in range(steps):
        x0 = few_hearts[torch.randint(20, (128,), device=device)]
        y = torch.full((128,), HEART, device=device)
        if replay:
            # Replace half the batch with freshly generated examples of the other classes
            other = torch.tensor([c for c in range(len(CLASSES)) if c != HEART])[torch.randint(len(CLASSES) - 1, (64,))]
            xo, _ = sample_shapes(64, classes=other)
            x0 = torch.cat([x0[:64], xo.to(device)]); y = torch.cat([y[:64], other.to(device)])
            y = torch.where(torch.rand(128, device=device) < 0.1, torch.full_like(y, model.null_class), y)
        t = torch.randint(0, T_shape, (128,), device=device)
        a = alpha_bars_s[t][:, None, None]; eps = torch.randn_like(x0)
        loss = ((model(a.sqrt() * x0 + (1 - a).sqrt() * eps, t, y) - eps) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return model

lora_model = finetune(make_lora_model())
show_all_classes(lora_model, 'LoRA, trained on hearts only')

**Things to look at and try**:

1. The model has learnt hearts from only 20 examples while training about 6% of its parameters. But look at the other shapes. What has happened, and why? (The LoRA updates change weights that every class uses.)
2. Run `finetune(make_lora_model(), replay=True)`, which mixes generated examples of the other classes into every batch. This is the same idea as DreamBooth's "prior preservation". Does it fix the problem?
3. Try rank 1 and rank 16. How few parameters can you get away with?
4. Compare with full fine-tuning (load `base_no_hearts.pt` into a fresh `ShapeDenoiser`, make everything trainable, and use `finetune(..., lr=2e-4)`). How many parameters are trained now?
5. Sample the hearts-only LoRA model *with* guidance, `show_all_classes(lora_model, 'w = 2', w=2)`. Why might guidance behave unpredictably after fine-tuning on hearts alone?

In [ ]:
# Solution
lora_replay = finetune(make_lora_model(), replay=True)
show_all_classes(lora_replay, 'LoRA with replay')

lora_r1 = finetune(make_lora_model(r=1), replay=True)
show_all_classes(lora_r1, 'LoRA rank 1 with replay')

full = ShapeDenoiser().to(device); full.load_state_dict(torch.load('base_no_hearts.pt'))
full = finetune(full, replay=True, lr=2e-4)
show_all_classes(full, 'Full fine-tuning with replay')